# LLM Finetuning Sprint

In [1]:
# 1. Clone the repository into the Colab environment
!git clone https://github.com/MircoFernando/Advanced-Hybrid-RAG-Finacial-Intelligence.git

Cloning into 'Advanced-Hybrid-RAG-Finacial-Intelligence'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 50 (delta 10), reused 47 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 8.24 MiB | 13.68 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [1]:
%cd Advanced-Hybrid-RAG-Finacial-Intelligence

/content/Advanced-Hybrid-RAG-Finacial-Intelligence


In [8]:
!git checkout data-factory

Branch 'data-factory' set up to track remote branch 'data-factory' from 'origin'.
Switched to a new branch 'data-factory'


## 1. Installing Dependencies

In [2]:
%%capture

# Core training libraries
!pip install -q \
    transformers==4.44.2 \
    datasets==2.20.0 \
    tokenizers==0.19.1 \
    accelerate==0.34.2 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.46.1 \
    evaluate==0.4.2

# Utilities
!pip install -q \
    numpy \
    pandas \
    scikit-learn \
    rich \
    pyyaml \
    python-dotenv \
    tqdm

# Evaluation
!pip install -q pydantic>=2.9 openai rouge-score

print("✅ Installation complete!")
print("✅ All dependencies compatible (pydantic v2 + openai)")

In [3]:
import numpy as np
print(np.__version__)

1.26.4


In [7]:
!pip install --upgrade numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 81.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.9.6 requires numpy<2.0.0,>=1.18.2, but you have numpy 2.4.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


## 2. Environment & GPU Check

In [4]:
import sys
import torch

print("="*60)
print("ENVIRONMENT CHECK")
print("="*60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ WARNING: CUDA not available. Training will be VERY slow on CPU.")

print("="*60)

ENVIRONMENT CHECK
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
Device name: Tesla T4
Device capability: (7, 5)
Total VRAM: 14.56 GB


## 3. Seeds & Determinism

In [5]:
import os
import random
import numpy as np
import torch

SEED = 42

# Set environment variable for Python hash seed
os.environ['PYTHONHASHSEED'] = str(SEED)

# Set seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Note: These settings may impact performance
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"✅ Seeds set to {SEED} for reproducibility")
print("⚠️ Note: Full determinism on GPU is not guaranteed due to non-deterministic operations")

✅ Seeds set to 42 for reproducibility
⚠️ Note: Full determinism on GPU is not guaranteed due to non-deterministic operations


## 4. Hugging Face Login

In [6]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!hf auth login --token $HF_TOKEN

print("ℹ️ Hugging Face login skipped. Uncomment login() to push models to Hub.")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `Mirco` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
ℹ️ Hugging Face login skipped. Uncomment login() to push models to Hub.


## 5. Configuration

In [7]:
from pathlib import Path
from pprint import pprint

# Auto-detect compute dtype (BF16 requires compute capability >= 8.0)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "artifacts").exists() and (PROJECT_ROOT.parent / "artifacts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

UBER_DATASET_PATH = PROJECT_ROOT / "artifacts" / "data_factory" / "uber_dataset.json"
TRAIN_JSONL_PATH = PROJECT_ROOT / "artifacts" / "train" / "train.jsonl"
GOLDEN_TEST_SET_PATH = PROJECT_ROOT / "artifacts" / "test" / "golden_test_set.jsonl"

CONFIG = {
    # Model
    "base_model": "meta-llama/Meta-Llama-3-8B-Instruct",

    # Local Uber data
    "train_files": [str(UBER_DATASET_PATH), str(TRAIN_JSONL_PATH)],
    "eval_file": str(GOLDEN_TEST_SET_PATH),
    "dataset_sample": 1500,

    # Tokenization
    "max_length": 512,

    # Training
    "num_train_epochs": 1,
    "max_steps": 200,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_steps": 50,
    "eval_steps": 50,
    "save_total_limit": 2,

    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],

    # Quantization
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": compute_dtype,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,

    # Output
    "output_dir": "outputs/adapter",
    "push_to_hub": False,

    # Generation
    "max_new_tokens": 128,
    "temperature": 0.0,
    "do_sample": False,
}

# Effective batch size
effective_batch_size = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]

print("="*60)
print("CONFIGURATION")
print("="*60)
pprint(CONFIG)
print("="*60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Uber dataset: {UBER_DATASET_PATH}")
print(f"Train JSONL: {TRAIN_JSONL_PATH}")
print(f"Golden test set: {GOLDEN_TEST_SET_PATH}")
print(f"Compute dtype: {compute_dtype}")
print(f"Using BF16: {use_bf16}")
print(f"Effective batch size: {effective_batch_size}")
print("="*60)

CONFIGURATION
{'base_model': 'meta-llama/Meta-Llama-3-8B-Instruct',
 'bnb_4bit_compute_dtype': torch.float16,
 'bnb_4bit_quant_type': 'nf4',
 'bnb_4bit_use_double_quant': True,
 'do_sample': False,
 'eval_file': '/content/Advanced-Hybrid-RAG-Finacial-Intelligence/artifacts/test/golden_test_set.jsonl',
 'eval_steps': 50,
 'gradient_accumulation_steps': 16,
 'learning_rate': 0.0002,
 'load_in_4bit': True,
 'logging_steps': 10,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'lora_r': 16,
 'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
 'max_length': 512,
 'max_new_tokens': 128,
 'max_steps': 120,
 'num_train_epochs': 1,
 'output_dir': 'outputs/adapter',
 'per_device_train_batch_size': 1,
 'push_to_hub': False,
 'save_steps': 50,
 'save_total_limit': 2,
 'temperature': 0.0,
 'train_files': ['/content/Advanced-Hybrid-RAG-Finacial-Intelligence/artifacts/data_factory/uber_dataset.json',
                 '/content/Advanced-Hybrid-RAG-Finacial-Intelligence/artifacts/train/train.js

## 6. Dataset Loading

In [1]:
from datasets import load_dataset, Dataset

# Load the dataset
def load_uber_dataset(path, sample):

  dataset = load_dataset("json", data_files=path)
  print(f"✅ Created synthetic dataset with {len(dataset)} examples")

  return dataset.shuffle().select(range(sample))

dataset = load_uber_dataset(str(TRAIN_JSONL_PATH), CONFIG["dataset_sample"])


NameError: name 'TRAIN_JSONL_PATH' is not defined

In [11]:
from transformers import AutoTokenizer
import numpy as np

# Load tokenizer for diagnostics
print(f"Loading tokenizer: {CONFIG['base_model']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"], trust_remote_code=True)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Sample up to 500 examples for diagnostics
sample_size = min(500, len(dataset))
sample_dataset = dataset.select(range(sample_size))

# Compute token lengths
token_lengths = []
for example in sample_dataset:
    # Concatenate instruction + input + output
    text = f"{example['question']} {example['answer']}"
    tokens = tokenizer(text, add_special_tokens=True)
    token_lengths.append(len(tokens["input_ids"]))

token_lengths = np.array(token_lengths)

print("="*60)
print("TOKEN LENGTH DIAGNOSTICS")
print("="*60)
print(f"Sample size: {sample_size}")
print(f"Average token length: {token_lengths.mean():.1f}")
print(f"Median token length: {np.median(token_lengths):.1f}")
print(f"Min token length: {token_lengths.min()}")
print(f"Max token length: {token_lengths.max()}")
print(f"95th percentile: {np.percentile(token_lengths, 95):.1f}")
print(f"99th percentile: {np.percentile(token_lengths, 99):.1f}")
print()
truncated = (token_lengths > CONFIG["max_length"]).sum()
truncation_rate = truncated / len(token_lengths) * 100
print(f"Truncation at max_length={CONFIG['max_length']}: {truncated}/{len(token_lengths)} ({truncation_rate:.1f}%)")
print("="*60)

Loading tokenizer: meta-llama/Meta-Llama-3-8B-Instruct


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
403 Client Error. (Request ID: Root=1-6a106024-3404af285f2477b175033501;9ddf41c6-7723-44f2-8f5f-af2c0a6c24e7)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct to ask for access.